# IMU session

The BNO085, and the three things it refuses over.

In [1]:
SIMULATED = True          # False, and PORT, at the bench
PORT = 'COM4'

In [2]:
from coaxial import Coaxial63100

device = Coaxial63100(port=PORT, simulated_device=SIMULATED).open()
print(device)

<Coaxial63100 Simulated SIMULATED>


The part is powered by AFE_ON. Off, it still answers reads, resets and advertises while acting on no write, so the front end goes up first.

In [3]:
from coaxial.errors import RigError

imu = device.imu
try:
    with imu.configuring():
        print(imu.product_id())
except RigError as exc:
    print('refused with the AFE off:', exc)
device.afe.enable()
print(device.afe.state())

{'reset_cause': 1, 'reset_cause_name': 'power on reset', 'sw_version': 'simulated', 'sw_part': 0, 'sw_build': 0, 'sw_patch': 0}
{'on': True, 'pe15': False, 'users': ['host']}


Every operation that drives SPI2 - `feature`, `product_id`, `reset`, `probe`, `write` - needs the poll loop held; the board refuses them while it runs, because both would be masters on one bus. `configuring()` holds and resumes.

In [4]:
try:
    print(imu.product_id())
except RigError as exc:
    print('refused while the loop runs:', exc)

with imu.configuring():
    print(imu.product_id())
    print(imu.pins())
    print('wake test:', imu.wake_test())

{'reset_cause': 1, 'reset_cause_name': 'power on reset', 'sw_version': 'simulated', 'sw_part': 0, 'sw_build': 0, 'sw_patch': 0}
{'reset_cause': 1, 'reset_cause_name': 'power on reset', 'sw_version': 'simulated', 'sw_part': 0, 'sw_build': 0, 'sw_patch': 0}
[{'pin': 'PB12', 'signal': 'NSS/H_CSN', 'bits': 15, 'held': False}, {'pin': 'PB13', 'signal': 'SCK', 'bits': 15, 'held': False}, {'pin': 'PB14', 'signal': 'MISO', 'bits': 15, 'held': False}, {'pin': 'PB15', 'signal': 'MOSI', 'bits': 15, 'held': False}]
wake test: 0


Set Feature: report 0x05 is the rotation vector, the interval in microseconds, 0 disables it. A write into a part still announcing itself after a reset is a write nobody acts on, which is why the firmware drains first.

In [5]:
import time

ROTATION_VECTOR = 0x05
with imu.configuring():
    imu.feature(ROTATION_VECTOR, 20000)
time.sleep(0.3)
for _ in range(5):
    st = imu.state()
    q = st['quaternion']
    print(st['loop'], st['updates'], st['feature'],
          None if q is None else {k: round(v, 3) for k, v in q.items()})
    time.sleep(0.1)

running 17 {'report_id': 5, 'interval_us': 20000, 'pending': False} {'i': 0.012, 'j': 0.025, 'k': -0.0, 'real': 1.0}
running 34 {'report_id': 5, 'interval_us': 20000, 'pending': False} {'i': 0.024, 'j': 0.049, 'k': -0.001, 'real': 0.998}
running 51 {'report_id': 5, 'interval_us': 20000, 'pending': False} {'i': 0.037, 'j': 0.073, 'k': -0.003, 'real': 0.997}


running 68 {'report_id': 5, 'interval_us': 20000, 'pending': False} {'i': 0.049, 'j': 0.098, 'k': -0.005, 'real': 0.994}
running 85 {'report_id': 5, 'interval_us': 20000, 'pending': False} {'i': 0.061, 'j': 0.122, 'k': -0.007, 'real': 0.991}


The three vectors ride the same reply since MINOR 6, each with its own `have`; a feature nobody enabled is None, not zero.

In [6]:
st = imu.state()
for name in ('accelerometer', 'gyroscope', 'magnetometer'):
    print('%-14s %s' % (name, st.get(name)))

accelerometer  None
gyroscope      None
magnetometer   None


In [7]:
with imu.configuring():
    imu.feature(ROTATION_VECTOR, 0)
device.close()

## Conclusions

In [8]:
print('loop            %s' % st['loop'])
print('updates         %d monotonic, cargoes %d, errors %d'
      % (st['updates'], st['cargoes'], st['errors']))
print('feature asked   report 0x%02X at %d us, pending %s'
      % (st['feature']['report_id'], st['feature']['interval_us'],
         st['feature']['pending']))
print('last fault      %s (id %d)' % (st['last_fault'], st['last_fault_id']))
print('quaternion      %s' % st['quaternion'])
for name in ('accelerometer', 'gyroscope', 'magnetometer'):
    print('%-15s %s' % (name, 'not enabled' if st.get(name) is None else st[name]['unit']))

loop            running
updates         102 monotonic, cargoes 102, errors 0
feature asked   report 0x05 at 20000 us, pending False
last fault      none (id 0)
quaternion      {'i': 0.07275390625, 'j': 0.14630126953125, 'k': -0.0107421875, 'real': 0.9864501953125}
accelerometer   not enabled
gyroscope       not enabled
magnetometer    not enabled


The three refusals:

1. **AFE_ON low.** The rail powers the part, not just the front end. Unpowered it still drives MISO, resets and advertises - a valid 276-byte advertisement reads back - while acting on no write, so every symptom presents as SPI. `Board_ImuInit` refuses while PB2 is low.
2. **The poll loop running.** Both driving SPI2 is two masters on one bus; `configuring()` holds and resumes.
3. **A part mid-sentence.** H_INTN stays asserted until everything queued is collected, so a write on top of a reset's three announcements loses both messages - SERVER DEVICE FAILURE. The firmware drains first, three empty reads a couple of milliseconds apart being quiet.

`updates` is monotonic, so the same reading read twice is telling. `error` is the last poll's and clears on the next good read; `last_fault` is what a host polling at 5 Hz would never see.